<a href="https://colab.research.google.com/github/Bukunmi2108/ml-journey/blob/main/research/p2/quant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Quantization from scratch

In [29]:
import torch
import torch.nn as nn

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [31]:
class INTQuantizer():
  @staticmethod
  def quantize_int8_per_tensor(tensor: torch.Tensor):
    scale = tensor.abs().max() / 127
    scale = scale.clamp(min=1e-8)
    out = torch.round(tensor / scale).clamp(-128, 127).to(torch.int8)
    return out, scale

  @staticmethod
  def dequantize_int8_per_tensor(q_tensor: torch.Tensor, scale: torch.Tensor):
    return q_tensor.float() * scale

  @staticmethod
  def dequantize_int8_per_channel(q_tensor: torch.Tensor, scales: torch.Tensor):
    return q_tensor.float() * scales

  @staticmethod
  def quantize_int8_per_channel(tensor: torch.Tensor): # quantizes/scales per row
    scales = tensor.abs().amax(dim=1, keepdim=True) / 127
    scales = scales.clamp(min=1e-8)
    out = torch.round(tensor / scales).clamp(-127, 127).to(torch.int8)
    return out, scales

Test INTQuantizer class

In [32]:
iq = INTQuantizer()
test_tensor = torch.tensor([-10.0, 0.0, 5.0, 10.0])

quantized_tensor, scale = iq.quantize_int8_per_tensor(test_tensor)
assert torch.isclose(scale, torch.tensor(10.0/127.0))

dequantized_tensor = iq.dequantize_int8_per_tensor(quantized_tensor, scale)
assert torch.allclose(test_tensor, dequantized_tensor, atol=scale.item()/2)

In [33]:
test_tensor = torch.tensor([[-10., 0., 5., 10.], [-20., -5., 0., 15.]])
quantized_tensor, scales = iq.quantize_int8_per_channel(test_tensor)
dequantized_tensor = iq.dequantize_int8_per_channel(quantized_tensor, scales)
max_atol = scales.max().item() / 2
assert torch.allclose(test_tensor, dequantized_tensor, atol=max_atol)